# Getting started with Claude Sonnet 4.6 on Amazon Bedrock

**Build high-performance AI applications with Anthropic's best balance of intelligence and speed**

This notebook demonstrates how to use Claude Sonnet 4.6 on Amazon Bedrock. As Anthropic's most capable model for everyday tasks, Sonnet 4.6 delivers strong performance at lower cost and latency — making it the right choice for customer-facing applications, high-throughput pipelines, and agentic workflows where cost efficiency matters.

---

## What you'll learn

- **Basic usage with Converse API**: Send single-turn and multi-turn conversations using Bedrock's unified API
- **Basic usage with InvokeModel API**: Call Claude directly using the native Anthropic request format
- **Adaptive thinking**: Enable on-demand reasoning with configurable effort levels — Claude decides whether to think based on query complexity
- **Prompt caching**: Mark system prompts and documents as cacheable to reduce cost and latency across repeated requests
- **Structured outputs**: Constrain responses to a JSON schema for type-safe, parse-error-free extraction
- **Tool use**: Define functions Claude can call for structured extraction, API integration, and agentic workflows
- **Vision**: Process images alongside text for document analysis, screenshot understanding, and multimodal tasks
- **Streaming**: Deliver tokens as they are generated to reduce perceived latency for long responses
- **Memory tool**: Persist and retrieve facts across sessions using client-side file operations
- **Context compaction**: Extend long-running conversations past the context window using server-side summarisation

---

## When to use Claude Sonnet 4.6

Claude Sonnet 4.6 is ideal for:

- **Customer-facing applications** requiring fast, high-quality responses
- **High-throughput pipelines** where cost and latency matter
- **Agentic workflows** with repeated tool calls and long context
- **Content generation** at scale — summaries, classification, extraction
- **Coding assistance** — code review, generation, and debugging

---

## Key capabilities

| Capability | Details |
|------------|--------|
| **Context window** | Up to 200K tokens |
| **Max output** | Up to 64K tokens |
| **Adaptive thinking** | Extended reasoning with configurable effort levels (low, medium, high) |
| **Prompt caching** | Cache up to 4 breakpoints to reduce cost and latency on repeated context |
| **Vision** | Image understanding via base64 or S3 URLs |

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install boto3 --upgrade

In [ ]:
import boto3
import json
from botocore.config import Config

# Configuration
REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"

# Config with extended timeout for thinking requests
CONFIG = Config(read_timeout=300, retries={"max_attempts": 1})

# Initialize session
session = boto3.Session()

# Initialize the Bedrock Runtime client
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=CONFIG,
)

print(f"✓ Region: {REGION}")
print(f"✓ Model: {MODEL_ID}")
print(f"✓ Client initialized successfully")

---

## 2. Basic usage with Converse API

The Converse API provides a unified interface for conversational AI across Amazon Bedrock models. This is the simplest way to get started.

In [ ]:
# Basic request using the Converse API
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What are three key considerations when designing a REST API for a high-traffic service?"
                }
            ],
        }
    ],
    inferenceConfig={"maxTokens": 1024},
)

response_text = response["output"]["message"]["content"][0]["text"]
print(response_text)
print(
    f"\nUsage: Input tokens={response['usage']['inputTokens']}, "
    f"Output tokens={response['usage']['outputTokens']}"
)

### Multi-turn conversation

The Converse API makes it easy to manage multi-turn conversations by passing the full message history:

In [ ]:
# Multi-turn conversation with the Converse API
conversation_history = []

def chat(user_message: str) -> str:
    """Send a message and maintain conversation history."""
    conversation_history.append(
        {"role": "user", "content": [{"text": user_message}]}
    )

    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=conversation_history,
        inferenceConfig={"maxTokens": 1024},
    )

    assistant_message = response["output"]["message"]
    conversation_history.append(assistant_message)

    return assistant_message["content"][0]["text"]


print("Turn 1:")
print(chat("I'm building a Python web scraper. What library should I use?"))

print("\nTurn 2:")
print(chat("How do I handle rate limiting with that library?"))

print(f"\nConversation has {len(conversation_history)} message(s)")

---

## 3. Basic usage with InvokeModel API

The InvokeModel API provides direct access to Claude's native request format and supports the full set of Sonnet 4.6 features.

In [ ]:
# Basic request using the InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Write a Python function that validates an email address using a regex pattern.",
                }
            ],
        }
    ],
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID, body=json.dumps(request_body)
)

response_body = json.loads(response["body"].read())
print(response_body["content"][0]["text"])
print(
    f"\nUsage: Input tokens={response_body['usage']['input_tokens']}, "
    f"Output tokens={response_body['usage']['output_tokens']}"
)

---

## 4. Adaptive thinking

Adaptive thinking gives Claude the freedom to reason when it determines a query warrants it. Instead of always thinking (which adds latency and cost), Claude decides intelligently based on query complexity.

### Effort levels

| Effort | Description |
|--------|-------------|
| **high** | Default. Deep reasoning on complex tasks. |
| **medium** | Balanced. May skip thinking for straightforward queries. |
| **low** | Minimal thinking, prioritizes speed and cost. |

> **Note:** The `max` effort level is exclusive to Claude Opus models.

In [ ]:
# Adaptive thinking with the Converse API
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "A company has 5 servers. Server A processes 30% of requests, "
                        "Server B processes 25%, C processes 20%, D processes 15%, and E processes 10%. "
                        "Each server has a 2% error rate except Server C which has a 5% error rate. "
                        "If a request results in an error, what is the probability it came from Server C?"
                    )
                }
            ],
        }
    ],
    inferenceConfig={"maxTokens": 8000, "temperature": 1},
    additionalModelRequestFields={
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": "high"},
    },
)

output_message = response["output"]["message"]
has_thinking = any(
    block.get("type") == "thinking" for block in output_message["content"]
)
print(f"Claude decided to think: {has_thinking}\n")

for block in output_message["content"]:
    if block.get("type") == "thinking":
        print("🧠 Thinking (first 300 chars):")
        print(block["thinking"][:300] + "...\n")
    elif block.get("text"):
        print("💬 Response:")
        print(block["text"])

print(
    f"\n📊 Usage: Input tokens={response['usage']['inputTokens']}, "
    f"Output tokens={response['usage']['outputTokens']}"
)

In [ ]:
# Adaptive thinking with InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 8000,
    "temperature": 1,
    "thinking": {"type": "adaptive"},
    "output_config": {"effort": "high"},
    "messages": [
        {
            "role": "user",
            "content": (
                "You have a list of n integers. Design an algorithm to find all pairs "
                "that sum to a target value k. Analyze the time and space complexity "
                "of your solution and explain any trade-offs."
            ),
        }
    ],
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID, body=json.dumps(request_body)
)
response_body = json.loads(response["body"].read())

has_thinking = any(
    block["type"] == "thinking" for block in response_body["content"]
)
print(f"Claude decided to think: {has_thinking}\n")

for block in response_body["content"]:
    if block["type"] == "thinking":
        print("🧠 Thinking (first 300 chars):")
        print(block["thinking"][:300] + "...\n")
    elif block["type"] == "text":
        print("💬 Response:")
        text = block["text"]
        print(text[:800] + "..." if len(text) > 800 else text)

### Comparing effort levels

Lower effort levels skip thinking for simpler queries and produce faster, cheaper responses:

In [ ]:
# Compare effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str) -> dict:
    """Test adaptive thinking at a specific effort level."""
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 4000,
        "temperature": 1,
        "thinking": {"type": "adaptive"},
        "output_config": {"effort": effort_level},
        "messages": [{"role": "user", "content": prompt}],
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(request_body)
    )
    response_body = json.loads(response["body"].read())

    has_thinking = False
    thinking_chars = 0
    response_chars = 0

    for block in response_body["content"]:
        if block["type"] == "thinking":
            has_thinking = True
            thinking_chars = len(block["thinking"])
        elif block["type"] == "text":
            response_chars = len(block["text"])

    return {
        "effort": effort_level,
        "thought": has_thinking,
        "thinking_chars": thinking_chars,
        "response_chars": response_chars,
    }


# Use a moderately complex prompt
prompt = (
    "Explain the trade-offs between using a relational database versus "
    "a document store for a social media application."
)

print("Testing effort levels on the same prompt...\n")
for effort in ["low", "medium", "high"]:
    result = test_effort_level(effort, prompt)
    print(
        f"Effort: {result['effort']:6} | Thought: {str(result['thought']):5} | "
        f"Thinking: {result['thinking_chars']:5} chars | "
        f"Response: {result['response_chars']:5} chars"
    )

print("\n✓ Higher effort levels trigger more thinking on complex prompts")
print("  Note: 'max' effort is exclusive to Claude Opus models.")

---

## 5. Prompt caching

Prompt caching reduces latency and cost when you repeatedly send the same large context (system prompts, documents, examples) alongside different user queries.

**How it works:**
- Mark content as cacheable with `"cache_control": {"type": "ephemeral"}`
- The first request writes the cache; subsequent requests read from it
- Cache reads are significantly cheaper and faster than processing the same tokens again
- Cache lifetime is 5 minutes (refreshed on each read)

**When to use it:**
- Large system prompts used across many requests
- Reference documents or codebases included with every query
- Few-shot examples that stay constant across a session

---

## 6. Structured outputs

Structured outputs constrain Claude's response to a JSON schema you define — eliminating parsing errors and enforcing type safety. Set `additionalProperties: false` in your schema (required for structured outputs to work).

**Two ways to use structured outputs:**
- **Converse API**: via `outputConfig.textFormat`
- **InvokeModel API**: via `output_config.format`

In [ ]:
# Structured outputs — Converse API: extract lead information from an email
extraction_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "Customer name"},
        "email": {"type": "string", "description": "Customer email address"},
        "plan_interest": {"type": "string", "description": "Product plan of interest"},
        "demo_requested": {"type": "boolean", "description": "Whether a demo was requested"},
    },
    "required": ["name", "email", "plan_interest", "demo_requested"],
    "additionalProperties": False,  # Required for structured outputs
}

response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Extract the key information from this email: "
                        "John Smith (john@example.com) is interested in our Enterprise plan "
                        "and wants to schedule a demo for next Tuesday at 2pm."
                    )
                }
            ],
        }
    ],
    inferenceConfig={"maxTokens": 512},
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {
                "jsonSchema": {
                    "schema": json.dumps(extraction_schema),
                    "name": "lead_extraction",
                    "description": "Extract lead information from customer emails",
                }
            },
        }
    },
)

result = json.loads(response["output"]["message"]["content"][0]["text"])
print("Converse API — extracted lead:")
print(json.dumps(result, indent=2))

# Structured outputs — InvokeModel API: sentiment analysis with enum constraints
sentiment_schema = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral", "mixed"],
        },
        "confidence": {"type": "number", "description": "Score 0–1"},
        "key_phrases": {"type": "array", "items": {"type": "string"}},
        "summary": {"type": "string"},
    },
    "required": ["sentiment", "confidence", "key_phrases", "summary"],
    "additionalProperties": False,
}

request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Analyze the sentiment of this review: "
                        "'I've been using this product for three months. The build quality exceeded "
                        "my expectations and support was great. Documentation could be better, "
                        "but overall a fantastic purchase.'"
                    ),
                }
            ],
        }
    ],
    "output_config": {"format": {"type": "json_schema", "schema": sentiment_schema}},
}

response = bedrock_runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(request_body))
result = json.loads(json.loads(response["body"].read())["content"][0]["text"])
print("\nInvokeModel API — sentiment analysis:")
print(f"  Sentiment:  {result['sentiment']} ({result['confidence']:.0%} confidence)")
print(f"  Key phrases: {', '.join(result['key_phrases'])}")
print(f"  Summary:    {result['summary']}")

In [ ]:
# Prompt caching example
# Simulate a large system prompt that would be reused across many requests
LARGE_SYSTEM_PROMPT = """
You are an expert software engineer specializing in Python. You help developers 
write clean, efficient, and well-tested code. You follow PEP 8 style guidelines 
and always consider edge cases.

When reviewing code, you check for:
- Correctness: Does the code do what it claims?
- Performance: Are there unnecessary loops or redundant computations?
- Readability: Are variable names clear? Is the code well-structured?
- Error handling: Are exceptions handled appropriately?
- Testing: Are there unit tests? Are edge cases covered?

You provide concrete, actionable feedback with code examples.
""" * 10  # Repeat to simulate a realistic large prompt

def ask_with_cache(user_question: str) -> dict:
    """Send a request with the system prompt marked for caching."""
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "system": [
            {
                "type": "text",
                "text": LARGE_SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},  # Mark for caching
            }
        ],
        "messages": [
            {"role": "user", "content": user_question}
        ],
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(request_body)
    )
    response_body = json.loads(response["body"].read())

    usage = response_body["usage"]
    return {
        "text": response_body["content"][0]["text"],
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "cache_creation_tokens": usage.get("cache_creation_input_tokens", 0),
        "cache_read_tokens": usage.get("cache_read_input_tokens", 0),
    }


# First request: writes to cache
print("Request 1 (cache write):")
result1 = ask_with_cache("What are the most common Python anti-patterns to avoid?")
print(f"  Input tokens:          {result1['input_tokens']}")
print(f"  Cache creation tokens: {result1['cache_creation_tokens']}")
print(f"  Cache read tokens:     {result1['cache_read_tokens']}")
print(f"  Response: {result1['text'][:200]}...\n")

# Second request: reads from cache
print("Request 2 (cache read):")
result2 = ask_with_cache("How do I write effective unit tests in Python?")
print(f"  Input tokens:          {result2['input_tokens']}")
print(f"  Cache creation tokens: {result2['cache_creation_tokens']}")
print(f"  Cache read tokens:     {result2['cache_read_tokens']}")
print(f"  Response: {result2['text'][:200]}...\n")

if result2['cache_read_tokens'] > 0:
    print(f"✓ Cache hit! {result2['cache_read_tokens']} tokens served from cache.")
else:
    print("Cache not hit yet — the cache may still be warming up. Try running again.")

### Caching document context

A common pattern is to cache a large reference document and ask multiple questions about it:

In [ ]:
# Cache a large document and ask multiple questions
REFERENCE_DOCUMENT = """
# Python Best Practices Guide

## Code Style
Follow PEP 8 for all Python code. Use 4 spaces for indentation. 
Lines should not exceed 88 characters (Black formatter default).
Use descriptive variable names: `user_count` not `n`, `is_active` not `flag`.

## Functions
Keep functions small and focused on a single responsibility.
Use type hints for all function signatures.
Write docstrings for public functions using Google or NumPy style.
Avoid mutable default arguments — use None and initialize inside the function.

## Error Handling  
Catch specific exceptions, never use bare `except:` clauses.
Use context managers (`with` statements) for resource management.
Raise exceptions with descriptive messages.
Log errors with sufficient context for debugging.

## Performance
Use list comprehensions and generator expressions over explicit loops when readable.
Profile before optimizing — don't guess at bottlenecks.
Use `collections.defaultdict` and `collections.Counter` for common patterns.
Cache expensive computations with `functools.lru_cache`.

## Testing
Write tests before or alongside code (TDD or concurrent).
Use pytest as the test framework. Aim for 80%+ line coverage.
Test edge cases: empty inputs, None values, boundary conditions.
Use fixtures for shared test setup, parametrize for data-driven tests.
""" * 5  # Repeat to create a realistically large document

questions = [
    "What does this guide say about variable naming?",
    "How should I handle exceptions according to this guide?",
    "What testing framework is recommended?",
]

print(f"Document size: {len(REFERENCE_DOCUMENT)} characters\n")

for i, question in enumerate(questions, 1):
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": REFERENCE_DOCUMENT,
                        "cache_control": {"type": "ephemeral"},
                    },
                    {"type": "text", "text": question},
                ],
            }
        ],
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID, body=json.dumps(request_body)
    )
    response_body = json.loads(response["body"].read())
    usage = response_body["usage"]

    cache_status = "WRITE" if usage.get("cache_creation_input_tokens", 0) > 0 else "READ"
    cached_tokens = usage.get("cache_creation_input_tokens", 0) or usage.get("cache_read_input_tokens", 0)

    print(f"Q{i} [{cache_status}]: {question}")
    print(f"    Cached tokens: {cached_tokens} | Answer: {response_body['content'][0]['text'][:150]}...\n")

---

## 7. Tool use

Tool use lets Claude call functions you define — useful for structured data extraction, API integration, and agentic workflows.

In [ ]:
# Tool use: extract structured data from unstructured text
tools = [
    {
        "name": "extract_contact",
        "description": "Extract contact information from text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Full name"},
                "email": {"type": "string", "description": "Email address"},
                "phone": {"type": "string", "description": "Phone number"},
                "company": {"type": "string", "description": "Company name"},
            },
            "required": ["name"],
        },
    }
]

request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "tools": tools,
    "tool_choice": {"type": "tool", "name": "extract_contact"},
    "messages": [
        {
            "role": "user",
            "content": "Hi, I'm Sarah Chen from Acme Corp. Reach me at sarah.chen@acme.com or (415) 555-0182.",
        }
    ],
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID, body=json.dumps(request_body)
)
response_body = json.loads(response["body"].read())

for block in response_body["content"]:
    if block["type"] == "tool_use":
        print("🔧 Extracted contact:")
        print(json.dumps(block["input"], indent=2))

---

## 8. Vision

Claude can process images alongside text — useful for document analysis, screenshot understanding, and multimodal workflows. Images can be passed as base64-encoded data or as S3 URLs.

In [ ]:
import urllib.request
import base64

# Fetch a sample image
image_url = "https://httpbin.org/image/png"
with urllib.request.urlopen(image_url) as r:
    image_data = base64.b64encode(r.read()).decode("utf-8")

request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 256,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": image_data,
                    },
                },
                {"type": "text", "text": "What does this image show? One sentence."},
            ],
        }
    ],
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID, body=json.dumps(request_body)
)
response_body = json.loads(response["body"].read())
print("Vision response:")
print(response_body["content"][0]["text"])

---

## 9. Streaming

Streaming delivers tokens as they are generated, reducing perceived latency for long responses. Use `invoke_model_with_response_stream` instead of `invoke_model`.

In [ ]:
import sys

request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [
        {
            "role": "user",
            "content": "List five best practices for designing resilient AWS architectures.",
        }
    ],
}

response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=MODEL_ID, body=json.dumps(request_body)
)

print("🌊 Streaming response:")
for event in response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    if chunk["type"] == "content_block_delta":
        delta = chunk["delta"]
        if delta["type"] == "text_delta":
            print(delta["text"], end="", flush=True)
print()  # newline at end

---

## 10. Memory tool

The memory tool lets Claude persist and retrieve information across sessions by reading and writing files in a `/memories` directory. It is a **client-side tool**: Claude makes file-operation tool calls (`view`, `create`, `str_replace`, `delete`) and your application executes them locally.

Enable it by adding `{"type": "memory_20250818", "name": "memory"}` to your tools list. Claude automatically checks its memory directory before starting tasks and updates it as it works.

> **Note:** Always use `pathlib.Path.resolve()` to validate that file paths stay within the memory directory.

In [ ]:
import pathlib, shutil

# Use resolve() so macOS /tmp → /private/tmp symlink is handled correctly
MEMORY_DIR = pathlib.Path("/tmp/claude_memories").resolve()
MEMORY_DIR.mkdir(exist_ok=True)


def safe_path(p: str) -> pathlib.Path:
    """Resolve path and reject any traversal outside MEMORY_DIR."""
    rel = pathlib.Path(p.lstrip("/"))
    candidate = (MEMORY_DIR / rel).resolve()
    if not str(candidate).startswith(str(MEMORY_DIR)):
        raise ValueError(f"Path traversal rejected: {p}")
    return candidate


def handle_memory_tool(tool_input: dict) -> str:
    """Execute memory tool commands (view / create / str_replace / delete)."""
    cmd = tool_input["command"]
    path = safe_path(tool_input.get("path", "/"))

    if cmd == "view":
        if not path.exists():
            return f"The path {tool_input['path']} does not exist."
        if path.is_dir():
            entries = sorted(path.rglob("*"))
            lines = [f"Contents of {tool_input['path']}:"]
            lines += [f"  {e.relative_to(MEMORY_DIR)}" for e in entries if e.is_file()]
            return "\n".join(lines) or "  (empty)"
        numbered = "\n".join(f"{i+1:>4}\t{l}" for i, l in enumerate(path.read_text().splitlines()))
        return f"{tool_input['path']}:\n{numbered}"

    elif cmd == "create":
        if path.exists():
            return f"Error: {tool_input['path']} already exists"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(tool_input.get("file_text", ""))
        return f"Created {tool_input['path']}"

    elif cmd == "str_replace":
        if not path.exists():
            return f"Error: {tool_input['path']} does not exist"
        old, new = tool_input["old_str"], tool_input["new_str"]
        content = path.read_text()
        if old not in content:
            return f"str not found: {old!r}"
        path.write_text(content.replace(old, new, 1))
        return "File updated."

    elif cmd == "delete":
        if not path.exists():
            return f"Error: {tool_input['path']} does not exist"
        shutil.rmtree(path) if path.is_dir() else path.unlink()
        return f"Deleted {tool_input['path']}"

    return f"Unknown command: {cmd}"


def run_memory_agent(user_message: str, history: list) -> str:
    """Agentic loop: drives Claude until end_turn, executing memory file ops locally."""
    history.append({"role": "user", "content": user_message})
    while True:
        body = json.loads(bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 1024,
                "tools": [{"type": "memory_20250818", "name": "memory"}],
                "messages": history,
            }),
        )["body"].read())

        history.append({"role": "assistant", "content": body["content"]})
        if body["stop_reason"] == "end_turn":
            return next(b["text"] for b in body["content"] if b["type"] == "text")

        tool_results = []
        for block in body["content"]:
            if block["type"] == "tool_use":
                cmd = block["input"].get("command", "?")
                p = block["input"].get("path", "")
                result = handle_memory_tool(block["input"])
                print(f"  🔧 memory.{cmd}({p})")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block["id"],
                    "content": result,
                })
        history.append({"role": "user", "content": tool_results})


# Turn 1: Claude checks memory, stores a fact
history = []
print("Turn 1 — store preferences:")
print(run_memory_agent(
    "Remember that my project is called 'Helios' and I prefer concise bullet-point responses.",
    history,
))

# Turn 2: new session — Claude reads memory to recall context
print("\nTurn 2 — new session, recall from memory:")
print(run_memory_agent("What do you know about my project and preferences?", []))

---

## 11. Context compaction

Context compaction is a server-side beta feature that automatically summarises older conversation content when approaching the context window limit. This extends effective context for long-running agentic workflows without any manual summarisation logic.

**How it works:**
1. Enable compaction by adding `compact_20260112` to `context_management.edits`
2. When input tokens exceed the trigger threshold, the API generates a summary and returns a `compaction` block
3. Append the full response (including the compaction block) to your messages array — the API automatically drops all content before the compaction block on subsequent requests

**Key parameters:**

| Parameter | Default | Notes |
|-----------|---------|-------|
| `type` | required | Must be `"compact_20260112"` |
| `trigger.value` | 150,000 tokens | Minimum 50,000 |
| `pause_after_compaction` | false | Stop after summary so you can inject content before continuing |

> Requires beta header: `"anthropic_beta": ["compact-2026-01-12"]`

In [ ]:
# Context compaction: enable server-side summarisation via compact-2026-01-12 beta
messages = []

def chat_with_compaction(user_message: str) -> str:
    """Send a message with compaction enabled. Automatically handles compaction blocks."""
    messages.append({"role": "user", "content": user_message})

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "anthropic_beta": ["compact-2026-01-12"],
            "max_tokens": 1024,
            "messages": messages,
            "context_management": {
                "edits": [
                    {
                        "type": "compact_20260112",
                        "trigger": {
                            "type": "input_tokens",
                            "value": 50000,  # Trigger at 50K tokens (minimum allowed)
                        },
                    }
                ]
            },
        }),
    )
    body = json.loads(response["body"].read())

    # Append full response including any compaction block
    messages.append({"role": "assistant", "content": body["content"]})

    # Report if compaction was triggered
    has_compaction = any(b.get("type") == "compaction" for b in body["content"])
    if has_compaction:
        print("  📦 Compaction triggered — older context was summarised")

    return next((b["text"] for b in body["content"] if b["type"] == "text"), "")


# Multi-turn conversation — compaction will trigger automatically once 50K tokens are reached
turns = [
    "We're building a Python microservice using FastAPI.",
    "We've decided to use PostgreSQL for the database and Redis for caching.",
    "What technologies have we chosen so far?",
]

for i, msg in enumerate(turns, 1):
    reply = chat_with_compaction(msg)
    print(f"Turn {i}: {reply[:300]}")
    print(f"  History: {len(messages)} message(s)\n")

print(f"✓ Compaction is transparent — the conversation continues normally after a compaction event.")

---

## Summary

Claude Sonnet 4.6 is Anthropic's best balance of intelligence, speed, and cost — the right choice for most production workloads.

**Key features covered:**

| # | Feature | Key mechanism |
|---|---------|--------------|
| 2 | Converse API | `bedrock_runtime.converse()` with `messages` |
| 3 | InvokeModel API | `bedrock_runtime.invoke_model()` with Anthropic request body |
| 4 | Adaptive thinking | `additionalModelRequestFields: {thinking: {type: "adaptive"}}` + `effort` |
| 5 | Prompt caching | `cache_control: {type: "ephemeral"}` on content blocks |
| 6 | Structured outputs | `outputConfig.textFormat` (Converse) or `output_config.format` (InvokeModel) |
| 7 | Tool use | `toolConfig.tools` (Converse) or `tools` (InvokeModel) |
| 8 | Vision | `content[].type: "image"` with base64 or S3 source |
| 9 | Streaming | `invoke_model_with_response_stream` + event stream iteration |
| 10 | Memory tool | `tools: [{type: "memory_20250818"}]` + client-side file handler |
| 11 | Context compaction | `context_management.edits: [{type: "compact_20260112"}]` + beta header |

### Choose the right model

| Use case | Recommended model |
|----------|-------------------|
| Customer-facing apps, high-throughput pipelines | **Sonnet 4.6** (this notebook) |
| Complex reasoning, long-horizon agentic tasks | Claude Opus 4.6 / 4.7 |
| Fast, lightweight tasks at lowest cost | Claude Haiku 4.5 |

### Resources

- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)
- [Anthropic Documentation](https://docs.anthropic.com/)
- [Prompt caching guide](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)
- [Memory tool guide](https://docs.anthropic.com/en/agents-and-tools/tool-use/memory-tool)
- [Context compaction guide](https://docs.anthropic.com/en/build-with-claude/compaction)